# 22 · Direct and self-coherent detection

A coherent receiver ([24](24-coherent-transceiver-end-to-end.ipynb)) beats
the signal with a local oscillator and recovers the full field.  A single
photodiode measures only the **intensity** `|x|²`: cheap, and the receiver
of short-reach links.  This notebook covers:

1. the **square law** and the bandwidth it doubles;
2. **intensity modulation and direct detection** (IM-DD) of unipolar PAM-4,
   and its sensitivity to the received power;
3. **self-coherent** detection: a carrier sent along with a QAM signal
   lets one photodiode recover the field, at the price of the
   signal-signal beat.

It builds on the noise and metrics of [06](06-awgn-and-performance-metrics.ipynb)
and the optical fields of [20](20-chromatic-dispersion.ipynb).  Functions
that model absolute power take it as an argument (`received_power_dbm`):
samples are dimensionless.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation, Rect
from commkit.backend import to_device

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()


def host(a):
    return to_device(a, "cpu")

## 1. The square law

`apply_photodetection` returns `|x|²` in the input's power units plus the
photocurrent's noise: the signal's own power sets the photocurrent
`I = R P`, with shot noise `q I f_s` and thermal noise `i_n² f_s / 2` per
sample.

Squaring multiplies the field by itself, so its spectrum is convolved with
itself: a field of band `±(1 + ρ) R_s / 2` gives an intensity of band
`±(1 + ρ) R_s`.  At 2 samples per symbol the field fits in the sampled
band but its intensity does not, and the part beyond `±R_s` folds back.
Below, the share of the intensity's power beyond `±R_s` for three
rolloffs, computed at 4 samples per symbol, and the spectrum for `ρ = 0.5`
with the band a 2-sample-per-symbol record holds.

In [ ]:
RS = 10e9
C16 = Constellation.qam(16)
for rolloff in (0.1, 0.5, 1.0):
    field = ck.generate(
        C16, num_symbols=2**15, symbol_rate=RS, sps=4, pulse=RRC(rolloff=rolloff),
        rng=1, device=DEVICE,
    ).replace(domain="OPT", center_frequency=193.4e12)  # fmt: skip
    intensity = ck.impairments.apply_photodetection(
        field, received_power_dbm=0.0, shot_noise=False
    )
    y = host(intensity.samples)
    spectrum = np.abs(np.fft.fft(y - y.mean())) ** 2
    f = np.fft.fftfreq(y.size, 1 / intensity.sampling_rate)
    beyond = spectrum[np.abs(f) > RS].sum() / spectrum.sum()
    print(f"rolloff {rolloff}: {100 * beyond:.2f} % of the intensity beyond ±Rs")
    if rolloff == 0.5:
        _, ax = ck.plotting.plot_psd(
            intensity, nperseg=1024, title="Intensity of 16-QAM, RRC 0.5", label="|x|²"
        )
        for edge in (-RS, RS):
            ax.axvline(edge, linestyle=":", color="gray")
        plt.show()

The intensity extends to `±(1 + ρ) R_s`, but only its tails lie beyond
`±R_s`: 0.08 % of the power for `ρ = 0.1`, 1.9 % for 0.5 and 4.1 % for 1,
an aliasing floor of -31, -17 and -14 dB.  A direct-detection receiver
samples the photocurrent at `2 (1 + ρ)` samples per symbol, or filters it
first, when that floor matters.

## 2. IM-DD with unipolar PAM-4

Intensity modulation puts the information on the power: four levels
`0, a, 2a, 3a` of a unipolar PAM-4 (`Constellation.pam(4, unipolar=True)`),
NRZ pulses, the field their square root.  At the receiver the photocurrent
is the transmitted level plus noise.  With thermal noise only, the noise is
Gaussian with a fixed variance and `awgn_ber` gives the BER at the Es/N0 the
received power sets (real samples: `Es/N0 = E[y²] / (2σ²)`).  Shot noise
grows with the level, most on the top one.

In [ ]:
pam4 = Constellation.pam(4, unipolar=True)
tx = ck.generate(pam4, num_symbols=2**15, symbol_rate=RS, sps=1, pulse=Rect(), rng=2,
                 device=DEVICE)  # fmt: skip
xp = tx.xp
optical = tx.replace(
    samples=xp.sqrt(tx.samples).astype(xp.complex64),
    domain="OPT",
    center_frequency=193.4e12,
)
i_n, responsivity = 20e-12, 0.8
powers = np.arange(-22.0, -15.0, 1.0)
measured = {"thermal": [], "thermal + shot": []}
theory = []
es = float(np.mean(host(tx.samples) ** 2))
for p in powers:
    for name, shot in (("thermal", False), ("thermal + shot", True)):
        rx = ck.impairments.apply_photodetection(
            optical, received_power_dbm=p, responsivity=responsivity,
            thermal_noise_density=i_n, shot_noise=shot, rng=3,
        )  # fmt: skip
        measured[name].append(ck.metrics.ber(rx))
    # thermal sigma in output units: i_n sqrt(fs/2) over the current per unit
    k = 1e-3 * 10 ** (p / 10) / float(np.mean(host(tx.samples)))
    sigma = i_n * np.sqrt(tx.sampling_rate / 2) / (responsivity * k)
    theory.append(ck.metrics.awgn_ber(pam4, esn0_db=10 * np.log10(es / (2 * sigma**2))))
fig, ax = plt.subplots()
ax.semilogy(powers, theory, color="gray", label="theory, thermal noise")
for (name, ber), marker in zip(measured.items(), ("o", "x"), strict=True):
    ax.semilogy(powers, np.where(np.array(ber) > 0, ber, np.nan), marker, label=name)
ax.set_xlabel("Received power [dBm]")
ax.set_ylabel("BER")
ax.set_title("PAM-4 IM-DD at 10 GBd, 20 pA/sqrt(Hz)")
ax.legend()
plt.show()
for p, a, b in zip(
    powers, measured["thermal"], measured["thermal + shot"], strict=True
):
    if p in (-20.0, -17.0):
        print(f"{p:.0f} dBm: thermal {a:.1e}, with shot noise {b:.1e}")

The thermal-noise points follow the closed form; shot noise costs little
at these powers, where the thermal term dominates.  Every 1 dB of received
power is 2 dB of electrical SNR: the photocurrent, not the field, carries
the signal.

## 3. Self-coherent detection

A carrier sent with the signal acts as a local oscillator that travels
with it.  With the field `c + s(t)` the photodiode sees

$$|c + s|^2 = |c|^2 + 2\,\mathrm{Re}\{c^* s\} + |s|^2 .$$

The middle term is the field itself, up to a known phase: shifted away
from the carrier by a **guard band**, the QAM signal lands at an
intermediate frequency, and shifting it back recovers it.  The last term,
the **signal-signal beat** (SSBI), occupies `±(1 + ρ) R_s` around DC.  The
carrier-to-signal power ratio (CSPR) sets how strong the wanted term is
against it.  Below, 16-QAM at 10 GBd, the carrier at DC, the signal's band
starting `guard` above it, a photodiode at -10 dBm with 10 pA/sqrt(Hz)
thermal noise, then DC removal, the shift back, the matched filter.

In [ ]:
qam = ck.generate(C16, num_symbols=2**15, symbol_rate=RS, sps=8,
                  pulse=RRC(rolloff=0.1), rng=4, device=DEVICE)  # fmt: skip
half_band = 0.55 * RS
print("CSPR   guard = 0   guard = 0.55 Rs   guard = 1.1 Rs   [SNR in dB]")
for cspr_db in (6.0, 10.0, 14.0):
    row = []
    for guard in (0.0, 0.55 * RS, 1.1 * RS):
        step = qam.sampling_rate / qam.samples.shape[-1]
        shift = round((guard + half_band) / step) * step  # on the FFT grid
        s = ck.spectral.shift_frequency(qam, frequency=shift)
        carrier = float(
            np.sqrt(np.mean(np.abs(host(s.samples)) ** 2) * 10 ** (cspr_db / 10))
        )
        sent = s.replace(samples=(s.samples + carrier).astype(xp.complex64),
                         domain="OPT", center_frequency=193.4e12)  # fmt: skip
        current = ck.impairments.apply_photodetection(
            sent, received_power_dbm=-10.0, thermal_noise_density=10e-12, rng=5
        )
        baseband = current.replace(
            samples=(current.samples - current.samples.mean()).astype(xp.complex64)
        )
        baseband = ck.spectral.shift_frequency(baseband, frequency=-shift)
        sym = ck.multirate.decimate_to_symbol_rate(
            ck.filtering.matched_filter(baseband)
        )
        row.append(ck.metrics.snr(sym, num_skip_symbols=100))
    print(f"{cspr_db:4.0f}  " + "  ".join(f"{v:12.2f}" for v in row))

Without a guard band the SSBI falls on the signal, and a stronger carrier
helps: the SNR rises with the CSPR.  A guard band as wide as the signal's
band moves the signal clear of it; then the limit is the thermal noise, and
at a fixed received power a stronger carrier leaves less power for the
signal, so the SNR falls with the CSPR.  Practical receivers sit between
the two: a smaller guard band with a digital SSBI cancellation, or the
Kramers-Kronig receiver, which needs no guard band at all when the carrier
is strong enough.

**Summary**

* A photodiode measures `|x|²`, whose band is twice the field's: sample or
  filter it accordingly.
* IM-DD of unipolar PAM-4 is limited by thermal noise at short reach; its
  BER against received power follows `awgn_ber` at the Es/N0 the
  photocurrent gives.
* A carrier sent with a QAM signal makes one photodiode a coherent
  receiver, traded against the signal-signal beat through the guard band
  and the CSPR.

**Next:** [23 · FEC in optical standards](23-fec-in-optical-standards.ipynb).